In [1]:
import os
import glob
import tgt
import pandas as pd
import pickle
import tqdm
import warnings
warnings.simplefilter("ignore")


import sys
sys.path.insert(0, "../../..")

from utils import data_processing
DATASET_PATH = "/Volumes/Transcend/FR-MRI/"

In [2]:
def read_misaligned_tgrd(f):    
    import tempfile
    with open(f) as file:
        lines = file.readlines()
        new_lines = []
        xmax = float(lines[4].split(" = ")[-1].replace("\n", ""))

        for line in lines:
            if line.startswith("\t\t\txmin"):
                this_xmin = float(line.split(" = ")[-1].replace("\n", ""))
                line = f"\t\t\txmin = {min(this_xmin, xmax)}\n"

            if line.startswith("\t\t\txmax"):
                this_xmax = float(line.split(" = ")[-1].replace("\n", ""))
                line = f"\t\t\txmax = {min(this_xmax, xmax)}\n"
            
            new_lines.append(line)

    with tempfile.TemporaryFile(mode='w+t', encoding='utf-8') as temp_file:
        temp_file.writelines(new_lines)
        temp_file.seek(0)
        tgrd = tgt.read_textgrid(temp_file)
        return tgrd

def read_tgrd(f):
    try:
        tgrd = tgt.read_textgrid(f)
    except ValueError:
        tgrd = read_misaligned_tgrd(f)
    return tgrd

In [3]:
tgrds = [(read_tgrd(f), f) for f in glob.glob(os.path.join(DATASET_PATH, "P*/OTHER/S*/*.textgrid"))]

In [4]:
def get_avi_path(tgrd_f):
    _, _, subject, session_id = os.path.basename(tgrd_f).replace(".textgrid", "").split("_")
    return f"/Volumes/Transcend/FR-MRI/{subject}/OTHER/{session_id}/VIDEO_{subject}_{session_id}.avi"

def get_wav_path(tgrd_f):
    _, _, subject, session_id = os.path.basename(tgrd_f).replace(".textgrid", "").split("_")
    return f"/Volumes/Transcend/FR-MRI/{subject}/OTHER/{session_id}/DENOISED_SOUND_{subject}_{session_id}.wav"

In [5]:
with open(
    "../../pronunciation_counts/pronunciation_dict_fr.pkl",
    "rb"
) as f:
    pronunciation_dict = pickle.load(f)

In [6]:
dfs_final = []

for tgrd, tgrd_f in tqdm.tqdm(tgrds):
    df_final = data_processing.alignment_to_dataframe(
        tgrd,
        lang="fr",
        pronunciation_dict=pronunciation_dict
    )
    art_f = get_avi_path(tgrd_f)
    ac_f = get_wav_path(tgrd_f)
    df_final["art_f"] = art_f
    df_final["ac_f"] = ac_f
    dfs_final.append(df_final)

100%|██████████| 160/160 [00:00<00:00, 175.60it/s]


In [7]:
df = pd.concat(dfs_final, ignore_index=True)

In [ ]:
df.to_csv("datasets/processed_frmri.csv", index=False)